# UMWP: exploratory analysis and data-quality validation

Inspect the official raw JSON Lines dataset before designing the external evaluation file.
This notebook is analysis only: it does not modify raw data, remove rows, create processed
files, split UMWP, or train models. Counts below describe the dataset as provided.

## 1. Setup and load dataset
Load each JSON Lines record with pandas. Resolve the project from either the project
root or the notebooks directory. Preserve category codes as objects so numeric codes
and nulls are retained without renaming.

In [1]:
from pathlib import Path
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "data/raw/umwp/StandardDataset.jsonl").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the project root or its notebooks directory.")
DATASET_PATH = PROJECT_ROOT / "data/raw/umwp/StandardDataset.jsonl"
df = pd.read_json(DATASET_PATH, lines=True, encoding="utf-8-sig",
                  dtype={"category": object})
required_columns = {"id", "question", "answerable", "category", "source", "answer", "relevant_ids"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"Missing required columns: {sorted(required_columns - set(df.columns))}")
print("Dataset path:", DATASET_PATH)
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
print("Column names:", df.columns.tolist())
display(df.head(5))


Dataset path: c:\Users\Murat Arikan\dev\school\03_hs_26\NLP\CDS1092_NLP_Project\data\raw\umwp\StandardDataset.jsonl
Rows: 5,200; columns: 7
Column names: ['id', 'question', 'answer', 'answerable', 'category', 'relevant_ids', 'source']


,id,question,answer,answerable,category,relevant_ids,source
0,1,Bryan took a look at his books and magazines. ...,[460.0],True,NaN,None,SVAMP
1,2,3 birds were sitting on the fence. 2 more bird...,[1.0],True,NaN,None,SVAMP
2,3,Randy has 58 blocks. He uses 27 blocks to buil...,[80.0],True,NaN,None,SVAMP
3,4,Dave had 17 apps and 21 files on his phone. Af...,[14.0],True,NaN,None,SVAMP
4,5,Haley grew 14 trees in her backyard. After a t...,[4.0],True,NaN,None,SVAMP


## 2. Dataset structure
Inspect every column and verify ID completeness and uniqueness.

In [2]:
display(df.dtypes.rename("dtype").to_frame())
df.info()
unique_ids = df["id"].nunique(dropna=True)
ids_unique = df["id"].is_unique and df["id"].notna().all()
print("Unique non-null IDs:", unique_ids)
print("Missing IDs:", int(df["id"].isna().sum()))
print("IDs unique and complete:", ids_unique)


,dtype
id,int64
question,object
answer,object
answerable,bool
category,object
relevant_ids,object
source,object


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            5200 non-null   int64 
 1   question      5200 non-null   object
 2   answer        2602 non-null   object
 3   answerable    5200 non-null   bool  
 4   category      2600 non-null   object
 5   relevant_ids  2600 non-null   object
 6   source        5200 non-null   object
dtypes: bool(1), int64(1), object(5)
memory usage: 249.0+ KB
Unique non-null IDs: 5200
Missing IDs: 0
IDs unique and complete: True


## 3. Missing values
Null `category` and `relevant_ids` values may be expected for answerable questions;
missingness alone is not an error. Later sections examine these fields by class.

In [3]:
missing = df.isna().sum().rename("missing_count").to_frame()
missing["missing_percent"] = df.isna().mean().mul(100).round(2)
display(missing)


,missing_count,missing_percent
id,0,0.00
question,0,0.00
answer,2598,49.96
answerable,0,0.00
category,2600,50.00
relevant_ids,2600,50.00
source,0,0.00


## 4. Class distribution
Check boolean labels and the expected 2,600 examples per class.

In [4]:
valid_labels = df["answerable"].map(lambda value: isinstance(value, bool))
class_counts = df.loc[valid_labels, "answerable"].value_counts().reindex([True, False], fill_value=0)
class_table = class_counts.rename_axis("answerable").rename("count").to_frame()
class_table["percent_of_all_rows"] = (class_table["count"] / len(df) * 100).round(2)
display(class_table)
print("Invalid or missing boolean labels:", int((~valid_labels).sum()))
expected_balance = class_counts.eq(2600).all() and valid_labels.all() and len(df) == 5200
print("Expected 2,600 answerable and 2,600 unanswerable:", bool(expected_balance))
answerable_mask = valid_labels & df["answerable"].eq(True)
unanswerable_mask = valid_labels & df["answerable"].eq(False)


,count,percent_of_all_rows
answerable,,
True,2600,50.0
False,2600,50.0


Invalid or missing boolean labels: 0
Expected 2,600 answerable and 2,600 unanswerable: True


## 5. Category analysis
Retain original category codes and inspect nulls separately by class.

In [5]:
display(df["category"].value_counts(dropna=False).rename("count").to_frame())
for label, mask in [(True, answerable_mask), (False, unanswerable_mask)]:
    print(f"Category distribution for answerable={label}:")
    display(df.loc[mask, "category"].value_counts(dropna=False).rename("count").to_frame())
answerable_null_categories = df.loc[answerable_mask, "category"].isna()
print("Answerable category null count:", int(answerable_null_categories.sum()))
print("Answerable category null percent:", round(answerable_null_categories.mean() * 100, 2))
unanswerable_categories = df.loc[unanswerable_mask, "category"].value_counts(dropna=False)
print("Unanswerable category counts (original values):")
display(unanswerable_categories.rename("count").to_frame())


,count
category,
NaN,2600
2.0,1263
1.0,840
3.0,275
5.0,119
4.0,103


Category distribution for answerable=True:


,count
category,
NaN,2600


Category distribution for answerable=False:


,count
category,
2.0,1263
1.0,840
3.0,275
5.0,119
4.0,103


Answerable category null count: 2600
Answerable category null percent: 100.0
Unanswerable category counts (original values):


,count
category,
2.0,1263
1.0,840
3.0,275
5.0,119
4.0,103


## 6. Source analysis
Compare class percentages within each source. A difference greater than 10 percentage
points from the overall answerable rate is flagged as a descriptive warning, not a
statistical significance test. Null sources remain visible in the tables.

In [6]:
print("Unique source values:", df["source"].unique().tolist())
source_counts = df["source"].value_counts(dropna=False).rename("count").to_frame()
source_counts["percent"] = (source_counts["count"] / len(df) * 100).round(2)
display(source_counts)
source_class_counts = pd.crosstab(df["source"], df["answerable"], dropna=False).reindex(
    columns=[True, False], fill_value=0)
print("Source x answerable (counts):")
display(source_class_counts)
source_class_percent = source_class_counts.div(source_class_counts.sum(axis=1), axis=0) * 100
print("Within-source class percentages:")
display(source_class_percent.round(2))
overall_answerable_percent = answerable_mask.sum() / valid_labels.sum() * 100
source_balance = source_class_percent.rename(columns={True: "answerable_percent"})[["answerable_percent"]]
source_balance["difference_from_overall_pp"] = source_balance["answerable_percent"] - overall_answerable_percent
source_balance["substantial_difference"] = source_balance["difference_from_overall_pp"].abs() > 10
display(source_balance.round(2))


Unique source values: ['SVAMP', 'MultiArith', 'GSM8K', 'ASDiv']


,count,percent
source,,
GSM8K,3400,65.38
SVAMP,1000,19.23
MultiArith,600,11.54
ASDiv,200,3.85


Source x answerable (counts):


answerable,True,False
source,,
ASDiv,100,100
GSM8K,1700,1700
MultiArith,300,300
SVAMP,500,500


Within-source class percentages:


answerable,True,False
source,,
ASDiv,50.0,50.0
GSM8K,50.0,50.0
MultiArith,50.0,50.0
SVAMP,50.0,50.0


answerable,answerable_percent,difference_from_overall_pp,substantial_difference
source,,,
ASDiv,50.0,0.0,False
GSM8K,50.0,0.0,False
MultiArith,50.0,0.0,False
SVAMP,50.0,0.0,False


## 7. Question quality checks
Separate missing, empty, whitespace-only, and non-string questions. Exact duplicates
use the original text, without trimming, lowercasing, or other normalization. Missing
and non-string values are excluded from text duplicate counts.

In [7]:
question_is_string = df["question"].map(lambda value: isinstance(value, str))
missing_questions = df["question"].isna()
empty_questions = df["question"].map(lambda value: isinstance(value, str) and value == "")
whitespace_questions = df["question"].map(
    lambda value: isinstance(value, str) and value != "" and value.strip() == "")
invalid_question_types = ~question_is_string & ~missing_questions
missing_or_empty_questions = missing_questions | empty_questions | whitespace_questions
question_counts = df.loc[question_is_string, "question"].value_counts()
duplicate_counts = question_counts[question_counts > 1]
duplicate_texts = duplicate_counts.index
duplicate_rows = df.loc[question_is_string & df["question"].isin(duplicate_texts)]
question_quality = pd.Series({
    "missing_questions": int(missing_questions.sum()),
    "empty_questions": int(empty_questions.sum()),
    "whitespace_only_questions": int(whitespace_questions.sum()),
    "non_string_non_null_questions": int(invalid_question_types.sum()),
    "unique_question_texts": len(question_counts),
    "duplicate_question_texts": len(duplicate_counts),
    "rows_involved_in_duplicates": int(duplicate_counts.sum()),
    "extra_duplicate_copies": int((duplicate_counts - 1).sum()),
}, name="count")
display(question_quality.to_frame())


,count
missing_questions,0
empty_questions,0
whitespace_only_questions,0
non_string_non_null_questions,0
unique_question_texts,5190
duplicate_question_texts,10
rows_involved_in_duplicates,20
extra_duplicate_copies,10


## 8. Label conflict check
Find exact question texts with multiple label values; retain null labels when checking.

In [8]:
duplicate_label_counts = duplicate_rows.groupby("question")["answerable"].nunique(dropna=False)
conflicting_texts = duplicate_label_counts[duplicate_label_counts > 1].index
conflict_rows = duplicate_rows.loc[duplicate_rows["question"].isin(conflicting_texts)]
print("Label-conflicting question texts:", len(conflicting_texts))
if len(conflicting_texts):
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(conflict_rows[["id", "question", "answerable", "category", "source"]]
                .sort_values(["question", "id"]))
else:
    print("No exact question text has conflicting labels.")


Label-conflicting question texts: 5


,id,question,answerable,category,source
429,430,A farmer had 175 tomatoes and 77 potatoes in his garden. If he picked 172 potatoes.How many tomatoes and potatoes does he have left?,True,NaN,SVAMP
3029,2930,A farmer had 175 tomatoes and 77 potatoes in his garden. If he picked 172 potatoes.How many tomatoes and potatoes does he have left?,False,3.0,SVAMP
472,473,"After eating a hearty meal they went to see the Buckingham palace. There, Rachel learned that 45 visitors came to the Buckingham palace on the previous day. If 829 people visited the Buckingham palace within the past 85 days.How many visitors visited the Buckingham palace on that day?",True,NaN,SVAMP
3072,2973,"After eating a hearty meal they went to see the Buckingham palace. There, Rachel learned that 45 visitors came to the Buckingham palace on the previous day. If 829 people visited the Buckingham palace within the past 85 days.How many visitors visited the Buckingham palace on that day?",False,2.0,SVAMP
212,213,"After eating a hearty meal they went to see the Buckingham palace. There, Rachel learned that 512 visitors came to the Buckingham palace that day. If there were 471 visitors the previous day and 808 visitors the day before that.How many visited the Buckingham palace within the past 89 days?",True,NaN,SVAMP
2812,2713,"After eating a hearty meal they went to see the Buckingham palace. There, Rachel learned that 512 visitors came to the Buckingham palace that day. If there were 471 visitors the previous day and 808 visitors the day before that.How many visited the Buckingham palace within the past 89 days?",False,2.0,SVAMP
1739,1740,"Molly got a bike for her thirteenth birthday. She rode her bike 3 miles a day, every day, until she turned 16. How many miles did Molly ride on her bike?",True,NaN,GSM8K
4339,4240,"Molly got a bike for her thirteenth birthday. She rode her bike 3 miles a day, every day, until she turned 16. How many miles did Molly ride on her bike?",False,1.0,GSM8K
856,857,Tate finishes high school in 1 year less than normal. It takes him 3 times that long to get his bachelor's degree and Ph.D. How many years did he spend in high school and college?,True,NaN,GSM8K
3456,3357,Tate finishes high school in 1 year less than normal. It takes him 3 times that long to get his bachelor's degree and Ph.D. How many years did he spend in high school and college?,False,1.0,GSM8K


## 9. Duplicate metadata inspection
Display all duplicate groups for manual review. No rows are removed.

In [9]:
if not duplicate_rows.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(duplicate_rows[["id", "question", "answerable", "category", "source"]]
                .sort_values(["question", "id"]))
else:
    print("No duplicate question groups to inspect.")


,id,question,answerable,category,source
611,612,A group of 3 friends went into a restaurant. The chef already had 6 chicken wings cooked but cooked 3 more for the group. If they each got the same amount how many would each person get?,True,NaN,MultiArith
671,672,A group of 3 friends went into a restaurant. The chef already had 6 chicken wings cooked but cooked 3 more for the group. If they each got the same amount how many would each person get?,True,NaN,MultiArith
511,512,A toy store had 6 giant stuffed bears in stock when they got another shipment with 18 bears in it. The put the bears onto shelves with 6 on each shelf. How many shelves did they use?,True,NaN,MultiArith
743,744,A toy store had 6 giant stuffed bears in stock when they got another shipment with 18 bears in it. The put the bears onto shelves with 6 on each shelf. How many shelves did they use?,True,NaN,MultiArith
628,629,"A vase can hold 5 flowers. If you had 6 carnations and 19 roses, how many vases would you need to hold the flowers?",True,NaN,MultiArith
776,777,"A vase can hold 5 flowers. If you had 6 carnations and 19 roses, how many vases would you need to hold the flowers?",True,NaN,MultiArith
562,563,"Haley and her friends were recycling paper for their class. For every 3 pounds they recycled they earned one point. If Haley recycled 11 pounds and her friends recycled 16 pounds, how many points did they earn?",True,NaN,MultiArith
733,734,"Haley and her friends were recycling paper for their class. For every 3 pounds they recycled they earned one point. If Haley recycled 11 pounds and her friends recycled 16 pounds, how many points did they earn?",True,NaN,MultiArith
604,605,"Jerry was helping the cafeteria workers pick up lunch trays, but he could only carry 8 trays at a time. If he had to pick up 9 trays from one table and 7 trays from another, how many trips will he make?",True,NaN,MultiArith
617,618,"Jerry was helping the cafeteria workers pick up lunch trays, but he could only carry 8 trays at a time. If he had to pick up 9 trays from one table and 7 trays from another, how many trips will he make?",True,NaN,MultiArith


## 10. Answer field analysis
Compare non-null answer availability by class. Non-null values do not guarantee a
usable mathematical answer; inspect value types and empty containers separately.
`answer` is metadata and must never be model input.

In [10]:
print("Missing answers:", int(df["answer"].isna().sum()))
answer_availability = pd.crosstab(df["answerable"], df["answer"].notna(), dropna=False).reindex(
    index=[True, False], columns=[False, True], fill_value=0)
answer_availability.columns = ["missing", "present"]
display(answer_availability)
display(answer_availability.div(answer_availability.sum(axis=1), axis=0).mul(100).round(2))
print("Answer value types:")
display(df["answer"].map(lambda value: type(value).__name__).value_counts().to_frame("count"))
print("Empty answer lists:", int(df["answer"].map(
    lambda value: isinstance(value, list) and len(value) == 0).sum()))
display(df[["id", "answerable", "answer"]].groupby("answerable", dropna=False).head(3))


Missing answers: 2598


,missing,present
answerable,,
True,0,2600
False,2598,2


,missing,present
answerable,,
True,0.00,100.00
False,99.92,0.08


Answer value types:


,count
answer,
list,2600
NoneType,2598
int,2


Empty answer lists: 0


,id,answerable,answer
0,1,True,[460.0]
1,2,True,[1.0]
2,3,True,[80.0]
2600,2501,False,None
2601,2502,False,None
2602,2503,False,None


## 11. Relevant IDs analysis
Inspect presence, types, and examples. `relevant_ids` must never be model input.

In [11]:
relevant_availability = pd.crosstab(df["answerable"], df["relevant_ids"].notna(), dropna=False).reindex(
    index=[True, False], columns=[False, True], fill_value=0)
relevant_availability.columns = ["missing", "present"]
display(relevant_availability)
display(relevant_availability.div(relevant_availability.sum(axis=1), axis=0).mul(100).round(2))
display(df["relevant_ids"].map(lambda value: type(value).__name__).value_counts().to_frame("count"))
print("Empty relevant_ids lists:", int(df["relevant_ids"].map(
    lambda value: isinstance(value, list) and len(value) == 0).sum()))
display(df[["id", "answerable", "relevant_ids"]].groupby("answerable", dropna=False).head(3))


,missing,present
answerable,,
True,2600,0
False,0,2600


,missing,present
answerable,,
True,100.0,0.0
False,0.0,100.0


,count
relevant_ids,
NoneType,2600
list,2600


Empty relevant_ids lists: 0


,id,answerable,relevant_ids
0,1,True,None
1,2,True,None
2,3,True,None
2600,2501,False,[1]
2601,2502,False,[2]
2602,2503,False,[3]


## 12. Final validation summary
Duplicate question count means distinct exact texts occurring more than once.
Warnings require review before processing; this notebook makes no cleaning decisions.

In [12]:
summary = pd.Series({
    "total_rows": len(df),
    "answerable_count": int(answerable_mask.sum()),
    "unanswerable_count": int(unanswerable_mask.sum()),
    "unique_questions": len(question_counts),
    "exact_duplicate_question_count": len(duplicate_counts),
    "label_conflicting_duplicate_count": len(conflicting_texts),
    "missing_or_empty_question_count": int(missing_or_empty_questions.sum()),
    "number_of_sources": df["source"].nunique(dropna=True),
}, name="value")
display(summary.to_frame())
print("Unanswerable category distribution:")
display(unanswerable_categories.rename("count").to_frame())

def check(condition, message):
    print(f"{'PASS' if condition else 'WARNING'}: {message}")

check(len(df) == 5200, "Expected 5,200 total rows")
check(expected_balance, "Expected 2,600 boolean labels per class")
check(ids_unique, "IDs are unique and non-null")
check(not missing_or_empty_questions.any(), "No missing, empty, or whitespace-only questions")
check(not invalid_question_types.any(), "All non-null questions are strings")
print(f"WARNING: {len(duplicate_counts)} exact duplicated question texts detected.")
print(f"WARNING: {len(conflicting_texts)} duplicated question texts have conflicting answerability labels.")
check(answerable_null_categories.all(), "Answerable categories are null (expected metadata convention)")
check(df.loc[unanswerable_mask, "category"].notna().all(), "Unanswerable examples have categories")
check(df["source"].notna().all(), "All rows have source metadata")
check(not source_balance["substantial_difference"].any(), "Source answerable rates within 10 percentage points of overall rate")
print("Analysis only: no rows removed, no processed dataset written, no model trained.")


,value
total_rows,5200
answerable_count,2600
unanswerable_count,2600
unique_questions,5190
exact_duplicate_question_count,10
label_conflicting_duplicate_count,5
missing_or_empty_question_count,0
number_of_sources,4


Unanswerable category distribution:


,count
category,
2.0,1263
1.0,840
3.0,275
5.0,119
4.0,103


PASS: Expected 5,200 total rows
PASS: Expected 2,600 boolean labels per class
PASS: IDs are unique and non-null
PASS: No missing, empty, or whitespace-only questions
PASS: All non-null questions are strings
PASS: Answerable categories are null (expected metadata convention)
PASS: Unanswerable examples have categories
PASS: All rows have source metadata
PASS: Source answerable rates within 10 percentage points of overall rate
Analysis only: no rows removed, no processed dataset written, no model trained.


## 13. Proposed processed schema

The processed UMWP external evaluation file will contain all 5,200 official rows with these columns:

| Field | Meaning |
| --- | --- |
| `id` | Original example identifier |
| `text` | Original `question` text |
| `label` | `answerable=True` maps to `ANSWERABLE`; `answerable=False` maps to `UNANSWERABLE` |
| `category` | Original category metadata for later error analysis |
| `source` | Source metadata for later generalization analysis |

Only `text` is model input. `label` is the ground truth used for evaluation.
`category` and `source` remain metadata; `answer`, `relevant_ids`, and other metadata are not model inputs.
UMWP will be used entirely as an external test/generalization dataset: no train/validation
split should be created for UMWP. Duplicate rows and conflicting labels are preserved
and documented, as explained in the decision below. This notebook does not create the processed evaluation file.

## Dataset Preparation Decision

UMWP is used only as an external evaluation/generalization benchmark, never for training
or hyperparameter selection. Encoder models are trained and selected using SUM only;
UMWP must not influence checkpoint selection or early stopping.

Exploratory analysis found 10 exact duplicated question texts; 5 of these occur with
conflicting answerability labels. Despite these inconsistencies, the official
5,200-instance dataset will be preserved unchanged: no duplicate rows will be removed,
and no labels will be corrected or changed. Preserving all official instances maintains
comparability with the official published UMWP benchmark.

Duplicate and conflicting examples will be considered explicitly during later error
analysis. In particular, predictions for identical question texts can be compared to
determine whether identical inputs receive identical predictions. Benchmark
inconsistencies should be distinguished from genuine model errors when interpreting results.
